<a href="https://colab.research.google.com/github/serdararici/btk-datathon-2026/blob/main/notebooks/v6_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datathon 2026 — v6 Final: Target Encoding + Soft-Blend Ceiling Correction

**Best Kaggle score:** Public 88.94 / Private 89.88

**Added in v6 (on top of v5):**
- Out-of-Fold Target Encoding for `department` and `target_role` (replaces One-Hot Encoding)
- Soft-Blend Ceiling Correction (handles 773 students with a clipped score of exactly 100)

**Kept from v5:** feature engineering, NLP, 3-model stacking (XGB+LGB+CAT), Ridge meta-model, Optuna-tuned hyperparameters

In [1]:
# ============================================================
# SECTION 0: SETUP
# ============================================================

!pip install catboost lightgbm -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_PATH    = '/content/drive/MyDrive/Colab Notebooks/btk-datathon-2026/'
DATASET_PATH = DATA_PATH + 'datasets/'
OUTPUT_PATH  = DATA_PATH + 'outputs/'

train = pd.read_csv(DATASET_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')

print(f"Train: {train.shape}, Test: {test.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.0 MB/s eta 0:00:00
Mounted at /content/drive
Train: (10000, 47), Test: (10000, 46)


In [2]:
# ============================================================
# SECTION 1: TEXT CLEANING + SENTIMENT FEATURES
# ============================================================

def clean_text(text):
    """Turkish-aware lowercasing: handles İ/I correctly (default .lower() does not)"""
    if pd.isnull(text):
        return ""
    text = text.replace('İ', 'i').replace('I', 'ı').lower()
    text = re.sub(r'[^a-zçğıöşü\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train['clean'] = train['mentor_feedback_text'].apply(clean_text)
test['clean']  = test['mentor_feedback_text'].apply(clean_text)

# Hand-built Turkish sentiment word lists
# Derived from manual inspection of mentor feedback texts
positive_words = ['dikkat','çekici','güçlü','başarı','başarılı','yetkin','yetenek','yetenekli',
    'uzmanlık','mükemmel','kanıtlıyor','sağlam','ileri','kayda','değer','avantaj','aranan',
    'kaliteli','üstün','parlak','etkileyici','olumlu','güzel','yüksek','kuvvetli','önemli']
negative_words = ['ancak','gereken','gerekiyor','gerekmekte','gerekebilir','gerekli','eksik',
    'zayıf','yetersiz','geliştirmesi','geliştirmek','çalışması','destek','gelişime','gelişmekte',
    'sorun','düşük','daha','fazla']
potential_words = ['potansiyel','potansiyeli','umut','verici','motivasyon','gelişim','çaba',
    'azim','azmi','hedef','hedefleri']

def count_words(text, word_list):
    return sum(1 for w in text.split() if w in word_list)

def add_sentiment(df):
    df['positive_count'] = df['clean'].apply(lambda x: count_words(x, positive_words))
    df['negative_count'] = df['clean'].apply(lambda x: count_words(x, negative_words))
    df['potential_count'] = df['clean'].apply(lambda x: count_words(x, potential_words))
    df['sentiment_score'] = df['positive_count'] - df['negative_count']
    df['sentiment_ratio'] = df['positive_count'] / (df['positive_count'] + df['negative_count'] + 1)
    return df

train = add_sentiment(train)
test  = add_sentiment(test)

print("Sentiment features added.")
print("sentiment_score correlation with target:",
      round(train['sentiment_score'].corr(train['career_success_score']), 4))

Sentiment features added.
sentiment_score correlation with target: 0.3981


In [3]:
# ============================================================
# SECTION 2: FEATURE ENGINEERING
# ============================================================

def create_features(df):
    """
    Derives new features from raw columns:
    - Skill aggregates (technical/soft averages)
    - Experience and ratio features
    - Interaction terms (capture combined effects)
    - cgpa interactions (cgpa has ~0 solo correlation but matters in interactions)
    - Log transforms for right-skewed count columns
    """
    df = df.copy()

    # Skill aggregates
    technical_cols = ['coding_score','problem_solving_score','data_structures_score',
                      'sql_score','machine_learning_score','backend_score',
                      'frontend_score','cloud_score','devops_score']
    df['avg_technical_score'] = df[technical_cols].mean(axis=1)

    soft_cols = ['communication_score','teamwork_score','leadership_score','presentation_score']
    df['avg_soft_skill_score'] = df[soft_cols].mean(axis=1)

    # Experience and ratio features
    df['experience_score'] = (df['internship_count'] * 3 + df['real_client_project_count'] * 2 +
                              df['freelance_project_count'] * 1 + df['hackathon_count'] * 1)
    df['interview_success_rate'] = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['hackathon_efficiency']   = df['hackathon_awards'] / (df['hackathon_count'] + 1)
    df['github_impact']          = df['github_repo_count'] * df['github_avg_stars'].fillna(0)
    df['total_project_count']    = df['real_client_project_count'] + df['freelance_project_count']

    # Interaction features
    df['interview_x_problem_solving'] = df['technical_interview_score'] * df['problem_solving_score']
    df['interview_x_cloud']           = df['technical_interview_score'] * df['cloud_score']
    df['interview_x_coding']          = df['technical_interview_score'] * df['coding_score']
    df['pq_x_interview']              = df['project_quality_score'] * df['technical_interview_score']

    # cgpa interactions (cgpa paradox: near-zero solo correlation, meaningful in interactions)
    df['cgpa_x_project_quality'] = df['cgpa'] * df['project_quality_score']
    df['cgpa_x_technical']       = df['cgpa'] * df['avg_technical_score']
    df['cgpa_x_interview']       = df['cgpa'] * df['technical_interview_score']

    # Log transforms for right-skewed count columns (reduces outlier influence)
    skewed_cols = ['github_avg_stars','github_repo_count','open_source_contribution_count',
                   'hackathon_count','certification_count','applications_sent']
    for col in skewed_cols:
        df[f'{col}_log'] = np.log1p(df[col].fillna(0))

    return df

train = create_features(train)
test  = create_features(test)
print(f"Feature engineering done. Train: {train.shape}, Test: {test.shape}")

Feature engineering done. Train: (10000, 73), Test: (10000, 72)


In [4]:
# ============================================================
# SECTION 3: PREPROCESSING (IMPUTATION + ENCODING)
# ============================================================

# Smart imputation for internship_duration_months:
# students with 0 internships logically have 0 duration (not truly missing)
dur_med = train.loc[train['internship_count'] > 0, 'internship_duration_months'].median()
for df in [train, test]:
    df.loc[(df['internship_count']==0) & (df['internship_duration_months'].isnull()),
           'internship_duration_months'] = 0
    df.loc[(df['internship_count']>0) & (df['internship_duration_months'].isnull()),
           'internship_duration_months'] = dur_med

# Median imputation using train-only statistics (prevents data leakage)
impute_cols = ['english_exam_score','portfolio_score','github_avg_stars',
               'open_source_contribution_count','linkedin_profile_score','hr_interview_score']
medians = {c: train[c].median() for c in impute_cols}
train.fillna(medians, inplace=True)
test.fillna(medians, inplace=True)

# university_tier is ordinal -> label encoding preserves the ranking
tier_map = {'Tier 1':4,'Tier 2':3,'Tier 3':2,'Tier 4':1}
train['university_tier'] = train['university_tier'].map(tier_map)
test['university_tier']  = test['university_tier'].map(tier_map)

print("Imputation and tier encoding done.")

Imputation and tier encoding done.


In [5]:
# ============================================================
# SECTION 3b: OUT-OF-FOLD TARGET ENCODING (new in v6)
# ============================================================
# Replaces One-Hot Encoding for department and target_role.
# Each category is replaced with the mean target score of that category,
# computed only from out-of-fold rows to prevent data leakage.

from sklearn.model_selection import KFold

kf_te = KFold(n_splits=5, shuffle=True, random_state=42)

for col in ['target_role', 'department']:
    train[col + '_target_enc'] = 0.0
    for tr_idx, vl_idx in kf_te.split(train):
        means = train.iloc[tr_idx].groupby(col)['career_success_score'].mean()
        global_mean = train.iloc[tr_idx]['career_success_score'].mean()
        train.loc[train.index[vl_idx], col + '_target_enc'] = (
            train.iloc[vl_idx][col].map(means).fillna(global_mean).values
        )
    # Test set uses full-train means (no leakage: test has no target values)
    full_means = train.groupby(col)['career_success_score'].mean()
    global_mean_full = train['career_success_score'].mean()
    test[col + '_target_enc'] = test[col].map(full_means).fillna(global_mean_full)

# Store target and ceiling mask before dropping the target column
target     = train['career_success_score'].copy()
is_ceiling = (target == 100).astype(int).values

# Drop noise columns, identifiers, raw text, and original categorical columns
drop_cols = ['hobby','preferred_social_media_platform','student_id',
             'mentor_feedback_text','clean','department','target_role']

train_clean = train.drop(columns=[c for c in drop_cols if c in train.columns] + ['career_success_score'])
test_clean  = test.drop(columns=[c for c in drop_cols if c in test.columns])
train_clean, test_clean = train_clean.align(test_clean, join='left', axis=1, fill_value=0)

print(f"Target encoding done. Train: {train_clean.shape}, Test: {test_clean.shape}")

Target encoding done. Train: (10000, 67), Test: (10000, 67)


In [6]:
# ============================================================
# SECTION 4: TF-IDF
# ============================================================
# Automatically selects the 100 most discriminative words/bigrams
# from mentor feedback texts (with Turkish stopword removal)

from sklearn.feature_extraction.text import TfidfVectorizer

turkish_stopwords = ['ve','ile','bir','bu','da','de','için','olan','olarak','çok','gibi',
    'kadar','sonra','daha','en','ya','veya','ama','ise','göre','her','ki','mi','ne','o','şu']

tfidf = TfidfVectorizer(max_features=100, stop_words=turkish_stopwords, ngram_range=(1,2))
tr_tf = tfidf.fit_transform(train['clean'])
te_tf = tfidf.transform(test['clean'])

tfidf_cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
tr_tf_df = pd.DataFrame(tr_tf.toarray(), columns=tfidf_cols)
te_tf_df = pd.DataFrame(te_tf.toarray(), columns=tfidf_cols)

X = pd.concat([train_clean.reset_index(drop=True), tr_tf_df], axis=1).astype(float)
X_test = pd.concat([test_clean.reset_index(drop=True), te_tf_df], axis=1).astype(float)
X = X.loc[:, ~X.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]

y = target.values
print(f"Final feature matrix: X={X.shape}, X_test={X_test.shape}")

Final feature matrix: X=(10000, 167), X_test=(10000, 167)


In [7]:
# ============================================================
# SECTION 5: 3-MODEL TRAINING + CEILING CLASSIFIER
# ============================================================
# Hyperparameters were found via Optuna (40 trials, 5-fold CV) in v5.
# They are reused here directly — no new search needed.

from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Hyperparameters from v5 Optuna tuning
xgb_params = dict(n_estimators=600, max_depth=5, learning_rate=0.035, subsample=0.8,
                   colsample_bytree=0.7, min_child_weight=4, reg_lambda=1.5,
                   random_state=42, n_jobs=-1)
lgb_params  = dict(n_estimators=700, max_depth=6, learning_rate=0.03, subsample=0.8,
                   colsample_bytree=0.7, num_leaves=31,
                   random_state=42, n_jobs=-1, verbose=-1)
cat_params  = dict(iterations=789, depth=6, learning_rate=0.0445, l2_leaf_reg=9.75,
                   random_state=42, verbose=0)

oof_xgb = np.zeros(len(y)); oof_lgb = np.zeros(len(y)); oof_cat = np.zeros(len(y))
oof_ceiling_prob = np.zeros(len(y))
test_xgb = np.zeros(len(X_test)); test_lgb = np.zeros(len(X_test)); test_cat = np.zeros(len(X_test))
test_ceiling_prob = np.zeros(len(X_test))

for fold, (tr, vl) in enumerate(kf.split(X)):
    Xtr, Xvl = X.iloc[tr], X.iloc[vl]
    ytr = y[tr]

    m_xgb = XGBRegressor(**xgb_params)
    m_xgb.fit(Xtr, ytr)
    oof_xgb[vl] = m_xgb.predict(Xvl)
    test_xgb += m_xgb.predict(X_test) / 5

    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(Xtr, ytr)
    oof_lgb[vl] = m_lgb.predict(Xvl)
    test_lgb += m_lgb.predict(X_test) / 5

    m_cat = CatBoostRegressor(**cat_params)
    m_cat.fit(Xtr, ytr)
    oof_cat[vl] = m_cat.predict(Xvl)
    test_cat += m_cat.predict(X_test) / 5

    # Ceiling classifier: estimates P(student scored exactly 100)
    # Used later in soft-blend correction
    clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                         random_state=42, n_jobs=-1, eval_metric='logloss')
    clf.fit(Xtr, is_ceiling[tr])
    oof_ceiling_prob[vl] = clf.predict_proba(Xvl)[:,1]
    test_ceiling_prob += clf.predict_proba(X_test)[:,1] / 5

    print(f"Fold {fold+1}/5 done")

print(f"\nXGBoost OOF MSE:  {mean_squared_error(y, oof_xgb):.2f}")
print(f"LightGBM OOF MSE: {mean_squared_error(y, oof_lgb):.2f}")
print(f"CatBoost OOF MSE: {mean_squared_error(y, oof_cat):.2f}")

Fold 1/5 done
Fold 2/5 done
Fold 3/5 done
Fold 4/5 done
Fold 5/5 done

XGBoost OOF MSE:  80.68
LightGBM OOF MSE: 81.15
CatBoost OOF MSE: 79.90


In [8]:
# ============================================================
# SECTION 6: RIDGE STACKING META-MODEL
# ============================================================
# Ridge learns how much to trust each base model from OOF predictions.
# CatBoost consistently receives the highest weight (~0.55)
# because it has the lowest OOF error.

from sklearn.linear_model import Ridge

meta_X      = np.column_stack([oof_xgb, oof_lgb, oof_cat])
meta_X_test = np.column_stack([test_xgb, test_lgb, test_cat])

meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X, y)

oof_stacked  = meta_model.predict(meta_X)
test_stacked = meta_model.predict(meta_X_test)

mse_stacked = mean_squared_error(y, oof_stacked)
print(f"Stacking OOF MSE: {mse_stacked:.2f}")
print(f"Ridge weights -> XGB: {meta_model.coef_[0]:.3f}, "
      f"LGB: {meta_model.coef_[1]:.3f}, "
      f"CAT: {meta_model.coef_[2]:.3f}, "
      f"intercept: {meta_model.intercept_:.3f}")

Stacking OOF MSE: 79.17
Ridge weights -> XGB: 0.248, LGB: 0.222, CAT: 0.553, intercept: -1.803


In [9]:
# ============================================================
# SECTION 6b: SOFT-BLEND CEILING CORRECTION (new in v6)
# ============================================================
# 773 students in the train set scored exactly 100 (clipped/censored target).
# Tree models systematically underpredict this group by ~4.7 points on average.
#
# Hard cutoff (e.g. "if pred > 94 then set to 100") was tested and rejected:
# false positives/negatives caused more damage than benefit.
#
# Soft-blend formula: predictions are pulled toward 100 proportionally
# to the ceiling classifier's confidence:
#   final = P(ceiling) * 100 + (1 - P(ceiling)) * stacked_prediction
#
# When the classifier is confident -> prediction moves close to 100
# When uncertain -> prediction stays close to the stacked regression output

oof_final  = oof_ceiling_prob * 100 + (1 - oof_ceiling_prob) * oof_stacked
test_final = test_ceiling_prob * 100 + (1 - test_ceiling_prob) * test_stacked

mse_final = mean_squared_error(y, oof_final)
print(f"FINAL OOF MSE (Target Encoding + Stacking + Soft-Blend): {mse_final:.2f}")
print(f"v5 reference OOF MSE: 79.41")
print(f"Kaggle Public MSE (verified): 88.94")

FINAL OOF MSE (Target Encoding + Stacking + Soft-Blend): 78.69
v5 reference OOF MSE: 79.41
Kaggle Public MSE (verified): 88.94


In [10]:
# ============================================================
# SECTION 7: SUBMISSION
# ============================================================

test_final_clipped = np.clip(test_final, 0, 100)

submission = pd.DataFrame({
    'student_id': pd.read_csv(DATASET_PATH + 'test_x.csv')['student_id'],
    'career_success_score': test_final_clipped
})
submission.to_csv(OUTPUT_PATH + 'submission_v6_final.csv', index=False)

print("Submission saved.")
print(f"Prediction mean:  {test_final_clipped.mean():.2f}")
print(f"Min: {test_final_clipped.min():.2f}, Max: {test_final_clipped.max():.2f}")
submission.head()

Submission saved.
Prediction mean:  76.48
Min: 33.82, Max: 100.00


,student_id,career_success_score
0,STU_010001,58.706091
1,STU_010002,76.936555
2,STU_010003,72.541078
3,STU_010004,98.097183
4,STU_010005,76.616714
